In [ ]:
## Fetching news articles via RSS feeds
import feedparser
from datetime import datetime
from transformers import pipeline
from newspaper import Article, Config

## rss feed list source: https://themeisle.com/blog/rss-feeds-list/#business

## bias source: [
#   https://www.allsides.com/media-bias/media-bias-chart,
#   https://adfontesmedia.com/  
#  ]
feeds = {
    # "Bloomberg" : { "url" : "https://www.bloomberg.com/feed/podcast/etf-report.xml", "bias" : "left" },
    "Forbes" : {"url" : "https://www.forbes.com/business/feed/", "bias" : "center"},
    # "CNBC" : {"url" : "https://www.cnbc.com/id/100003114/device/rss/rss.html", "bias" : "left"},
    # "Financial Times" : {"url" : "https://www.ft.com/?format=rss", "bias" : "center"}, 
    # "The Economist" : {"url" : "https://www.economist.com/latest/rss.xml", "bias" : "left"}, 
    # "Harvard Business Reviews" : {"url" : "https://hbr.org/feed", "bias" : "center"}, 
    # "Market Watch" : {"url" : "https://www.marketwatch.com/rss/topstories", "bias" : "center"}, 
    # "Wall Street Journal - Business" : {"url" : "https://feeds.a.dj.com/rss/RSSMarketsMain.xml", "bias" : "center"}, 
    # "Business Insider" : {"url" : "https://www.businessinsider.com/rss", "bias" : "center"}, 
    # "Investopedia" : {"url" : "https://www.investopedia.com/feedbuilder/feed/getfeed/?feedName=rss_articles", "bias" : "center"} 
}

## fetch articles from the rss feeds of various sources
articles = []
for source, info in feeds.items():
    feed = feedparser.parse(info["url"])
    for entry in feed.entries:  
        articles.append({
            "title": entry.title,
            "url": entry.link,
            "published_date": getattr(entry, "published", None),
            "source_name": source,
            "bias": info["bias"]
        })

print(articles)

## load the LLM for summarization
summarizer = pipeline("summarization", model="../outputs/fine-tuned-bart", tokenizer="../outputs/fine-tuned-bart/", framework="pt")

## function to generate summary for the body of text
def summarize_text(body, max_chunk_length=1024):
    # split the body into chunks
    chunks = [body[i:i+max_chunk_length] for i in range(0, len(body), max_chunk_length)]

    summaries = []
    for chunk in chunks:
        summary = summarizer(chunk, max_length = 60, min_length=30, do_sample=False)[0]['summary_text']
        summaries.append(summary)

    return " ".join(summaries)

## generate summary for each article
def process_article(article, text, image):
    body = text

    summary = summarize_text(body)
    return {
        "title": article["title"],
        "body": body,
        "summary": summary,
        "category": "Finance",
        "image_url": image,
        "left_bias": ,
        "right_bias": ,
        "center_bias": , 
        "source_url": article["url"],
        "source_name": article["source_name"],
        "published_date": article["published_date"]
    }

config = Config()
config.browser_user_agent = (
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
    "AppleWebKit/537.36 (KHTML, like Gecko) "
    "Chrome/120.0.0.0 Safari/537.36 Edg/120.0.0.0"
)
config.request_timeout = 10

updatedArticles = []
for article in articles:
    # for each article, extract the article body
    url = article["url"]    
    news = Article(url, config=config)
    news.download()
    news.parse()

    print(news.text)

    formatted_article = process_article(article, news.text, news.top_image)
    updatedArticles.append(formatted_article)

print(updatedArticles)



[{'title': 'CB Isaiah Rodgers Delivers Record Performance For Vikings In Week 3 Victory', 'url': 'https://www.forbes.com/sites/stevesilverman/2025/09/22/cb-isaiah-rodgers-delivers-record-performance-for-vikings-in-week-3-victory/', 'published_date': 'Mon, 22 Sep 2025 01:05:42 -0400', 'source_name': 'Forbes', 'bias': 'center'}, {'title': 'The Directions Of Mets And Yankees Were On Full Display As Summer Ends', 'url': 'https://www.forbes.com/sites/larryfleisher/2025/09/22/the-directions-of-mets-and-yankees-were-on-full-display-as-summer-ends/', 'published_date': 'Mon, 22 Sep 2025 00:32:31 -0400', 'source_name': 'Forbes', 'bias': 'center'}, {'title': 'China’s Impersonation Hack Strikes At The Heart Of How Washington Works', 'url': 'https://www.forbes.com/sites/ronakdesai/2025/09/22/chinas-impersonation-hack-strikes-at-the-heart-of-how-washington-works/', 'published_date': 'Mon, 22 Sep 2025 00:14:46 -0400', 'source_name': 'Forbes', 'bias': 'center'}, {'title': 'Final Trailer For ‘Wicked: F

Your max_length is set to 60, but you input_length is only 17. You might consider decreasing max_length manually, e.g. summarizer('...', max_length=8)


John Moolenaar, chairman of the House Select Committee on the Strategic Competition between the United States and the Chinese Communist Party, speaks at the Hill & Valley Forum 2025 at the U.S. Capitol Visitor Center. Tasos Katopodis/Getty Images

Earlier this month, the bipartisan House Select Committee on the Strategic Competition between the United States and the Chinese Communist Party disclosed that hackers linked to Beijing tried to impersonate its chairman, Representative John Moolenaar (R-Michigan). Using his name and title, the attackers sent emails of supposed draft American sanctions legislation to various stakeholders: prominent Washington law firms, business and trade associations, think tanks and at least one foreign government.

But the hackers were trying to get into the targets’ computer systems.

Investigators believe the campaign began earlier in the summer. One so-called “lure” sent this past July contained malware attributed to APT41, a prolific Chinese espionage g

Your max_length is set to 60, but you input_length is only 13. You might consider decreasing max_length manually, e.g. summarizer('...', max_length=6)


"Wicked: For Good" partial poster featuring Ariana Grande and Cynthia Erivo. Universal Pictures

The countdown to the release of Wicked: For Good, starring Cynthia Erivo and Ariana Grande, continues this week with the release of the final trailer to part two of the big-screen adaptation of the iconic 2003 Broadway musical.

The first film, Wicked, debuted in theaters on Nov. 22, 2024. The movie musical — which stars Erivo as Elphaba, who is destined to become the Wicked Witch of the West and Grande as Glinda, the future Good Witch of the North — went on to earn $473.2 million domestically and nearly $282.3 million internationally for a worldwide box office tally of $755.5 million.

Forbes Photos: Broadway’s Original ‘Wicked’ Stars Then And Now

On top of the film’s financial success, Wicked went on to be nominated for 10 Oscars — including Best Picture, Best Actress for Erivo and Best Supporting Actress for Grande — and won two, for Best Production Design for Nathan Crowley and Lee San

...

BEVERLY HILLS, CALIFORNIA - MAY 02: Jimmy Kimmel attends the 28th Annual UCLA Jonsson Cancer Center Foundation's "Taste For A Cure" event at Beverly Wilshire, A Four Seasons Hotel on May 02, 2025, in Beverly Hills, California. (Photo by Tommaso Boddi/Getty Images for UCLA Jonsson Cancer Center Foundation) Getty Images for UCLA Jonsson Cancer Center Foundation

A petition campaign from MoveOn is seeing a surge in sign-ups in response to ABC’s suspension of late-night host Jimmy Kimmel. In fact, within just 24 hours of launching, the petition drew more than 100,000 signatures from supporters demanding that ABC and Disney reinstate Kimmel — putting it in striking distance of being MoveOn’s fastest-growing petition of 2025 (as of Sunday afternoon, the petition had garnered around 154,000 signatures).

The effort comes after ABC abruptly pulled Jimmy Kimmel Live! off the air last week following threats from the Trump administration. Kimmel drew that ire after criticizing conservatives’ resp

Your max_length is set to 60, but you input_length is only 13. You might consider decreasing max_length manually, e.g. summarizer('...', max_length=6)


Green Bay quarterback Jordan Love (10) and the Packers suffered a horrendous 13-10 loss to Cleveland Sunday. Copyright 2025 The Associated Press. All rights reserved.

Earlier this week, Green Bay Packers left tackle Rasheed Walker said: “I think we can go undefeated, honestly.”

It’s amazing how fast things can change in the National Football League.

Green Bay had one of its worst performances of the Matt LaFleur-era Sunday and suffered a shocking 13-10 loss to Cleveland.

Browns kicker Andre Szmyt drilled a 55-yard field goal as time expired to lift Cleveland to victory. Just seconds earlier, Cleveland blocked a 43-yard field goal attempt by Brandon McManus that would have given the Packers the lead.

Green Bay fell to 2-1, while the Browns improved to 1-2.

Here’s the ‘Good, Bad and Ugly’ from Green Bay’s win, beginning with the UGLY.

THE UGLY

NO PROTECTION: Green Bay’s offensive line was atrocious, and quarterback Jordan Love was sacked five times and pressured several more.

Ri

Your max_length is set to 60, but you input_length is only 18. You might consider decreasing max_length manually, e.g. summarizer('...', max_length=9)


[{'title': 'CB Isaiah Rodgers Delivers Record Performance For Vikings In Week 3 Victory', 'summary': 'Isaiah Rodgers celebrates after scoring a touchdown off of a fumble recovery during the second quarter against the Cincinnati Bengals (Photo by Adam Bettcher/Getty Images) Getty Images/iStockphotoGoing into Sunday’s Week 3 game between the Vikings and the Bengals, the story line concerned k off immediately and sped down the sideline with sprinter’s speed and raced into the endzone. (Ph)That might have been enough to earn him defensive player of the week, but it was just the first salvo. Late in the second quarter with the Vikings leading oto by Adam Bettcher/Getty Images) Getty ImagesThe Vikings went into the locker room at halftime with a 34-3 lead. When it was all over, they had pummeled the Bengals by a 48-10 margin. Rodgers was the hero of the game for the Vikings o that,” Smith said, per The Athletic. “You’re usually just, like, ‘Can I make this tackle?’ The awareness to be able t